In [21]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

raw_corpus = {
    "China": "China, officially the People's Republic of China (PRC), is a vast country located in East Asia. It is the world's second-most populous country, with a population exceeding 1.4 billion. Beijing is the national capital, while Shanghai is the most populous city and largest financial center. China's landscape is vast and diverse, ranging from the Gobi and Taklamakan Deserts in the arid north to the subtropical forests in the wetter south. The Yangtze and Yellow Rivers, the third- and sixth-longest in the world, respectively, flow from the Tibetan Plateau to the densely populated eastern seaboard. China is a unitary one-party socialist republic and is recognized as a major global power, boasting the world's second-largest economy by nominal GDP. The Great Wall of China is one of its most famous historical landmarks.",
    
    "Russia": "Russia, officially the Russian Federation, is a transcontinental country spanning Eastern Europe and Northern Asia. It is the largest country in the world by land area, covering over 17 million square kilometers, and encompasses eleven time zones. Moscow is the capital and largest city, followed by Saint Petersburg, which serves as Russia's cultural center. The country's landscape includes vast plains, dense forests known as the taiga, and prominent mountain ranges such as the Urals and the Caucasus. Russia has a rich history characterized by the Russian Empire and its later role as the leading constituent of the Soviet Union. Today, it is a federal semi-presidential republic. The Russian economy is heavily reliant on its extensive natural resources, particularly oil and natural gas.",
    
    "Pakistan": "Pakistan, officially the Islamic Republic of Pakistan, is a country in South Asia. It is the world's fifth-most populous country, with a population of over 240 million people, and has the second-largest Muslim population. Islamabad is the nation's capital, while Karachi is its largest city and financial hub. Pakistan has a 1,046-kilometre coastline along the Arabian Sea and the Gulf of Oman in the south. The country's geography is highly diverse, featuring the towering peaks of the Karakoram and Himalayan mountain ranges in the north, including K2, the world's second-highest mountain. The Indus River flows through the length of the country, supporting agriculture and densely populated valleys. Pakistan was created in 1947 as an independent homeland for Indian Muslims following the partition of British India."
}

documents = list(raw_corpus.values())
columns_names = list(raw_corpus.keys())

vectorizer = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorizer.fit_transform(documents)

feature_names = vectorizer.get_feature_names_out()

df_tfidf = pd.DataFrame(tfidf_matrix.T.toarray(), index=feature_names, columns=columns_names)

df_sorted_by_china = df_tfidf.sort_values(by="China", ascending=False)
df_sorted_by_Russia = df_tfidf.sort_values(by="Russia", ascending=False)
df_sorted_by_Pakistan = df_tfidf.sort_values(by="Pakistan", ascending=False)
pd.set_option('display.max_rows', None)

print("TF-IDF matrix after cleaning and filtering stop words")
print(df_tfidf)
print("\n====== China top5 word ======")
print(df_sorted_by_china.head(5))
print("\n====== Russia top5 word ======")
print(df_sorted_by_Russia.head(5))
print("\n====== Pakistan top5 word ======")
print(df_sorted_by_Pakistan.head(5))

TF-IDF matrix after cleaning and filtering stop words
                     China    Russia  Pakistan
046               0.000000  0.000000  0.110046
17                0.000000  0.109875  0.000000
1947              0.000000  0.000000  0.110046
240               0.000000  0.000000  0.110046
agriculture       0.000000  0.000000  0.110046
arabian           0.000000  0.000000  0.110046
area              0.000000  0.109875  0.000000
arid              0.106491  0.000000  0.000000
asia              0.062895  0.064894  0.064995
beijing           0.106491  0.000000  0.000000
billion           0.106491  0.000000  0.000000
boasting          0.106491  0.000000  0.000000
british           0.000000  0.000000  0.110046
capital           0.062895  0.064894  0.064995
caucasus          0.000000  0.109875  0.000000
center            0.080989  0.083562  0.000000
characterized     0.000000  0.109875  0.000000
china             0.532455  0.000000  0.000000
city              0.062895  0.064894  0.064995
coastl

In [23]:
new_text = "This colossal nation spans two different continents, making it a bridge between distinct cultural spheres. Because of its sheer size, traveling from its western borders to its eastern shores means passing through multiple time zones. The natural environment is notoriously harsh, featuring vast, freezing plains and endless stretches of dense, needle-leaf forests. Historically, it has transitioned through powerful imperial eras and a major twentieth-century ideological union, leaving a complex legacy. Today, its global influence is heavily sustained by the extraction and exportation of immense underground energy reserves, particularly fossil fuels, which lie hidden beneath its freezing terrain."
new_vector = vectorizer.transform([new_text])

similarity_scores = cosine_similarity(new_vector, tfidf_matrix)[0]

results = dict(zip(columns_names, similarity_scores))

print("similarity score:")
for country, score in results.items():
    print(f" {country}: {score:.4f}")

best_match = max(results, key=results.get)
print(f"\n The country closest to this new text is: {best_match}")

new_text_nonzero_indices = new_vector.nonzero()[1]

russia_index = columns_names.index("Russia")
russia_vector = tfidf_matrix[russia_index]
russia_nonzero_indices = russia_vector.nonzero()[1]

shared_indices = np.intersect1d(new_text_nonzero_indices, russia_nonzero_indices)

evidence_data = []
for idx in shared_indices:
    word = feature_names[idx]
    weight_in_new = new_vector[0, idx]
    weight_in_russia = russia_vector[0, idx]
    evidence_data.append({
        "Matched resonance words": word,
        "Weight in new text": round(weight_in_new, 4),
        "Weight in the original Russian text": round(weight_in_russia, 4)
    })

df_evidence = pd.DataFrame(evidence_data).sort_values(by="Weight in the original Russian text", ascending=False)
print(df_evidence.to_string(index=False))


similarity score:
 China: 0.1158
 Russia: 0.3528
 Pakistan: 0.0555

 The country closest to this new text is: Russia
Matched resonance words  Weight in new text  Weight in the original Russian text
                natural              0.2521                               0.2197
                  dense              0.2521                               0.1099
                heavily              0.2521                               0.1099
           particularly              0.2521                               0.1099
               cultural              0.2521                               0.1099
                  today              0.2521                               0.1099
                  union              0.2521                               0.1099
                 plains              0.2521                               0.1099
                   time              0.2521                               0.1099
                  zones              0.2521                              

In [4]:
import math
import re

raw_corpus = {
    "China": "China, officially the People's Republic of China (PRC), is a vast country located in East Asia. It is the world's second-most populous country, with a population exceeding 1.4 billion. Beijing is the national capital, while Shanghai is the most populous city and largest financial center. China's landscape is vast and diverse, ranging from the Gobi and Taklamakan Deserts in the arid north to the subtropical forests in the wetter south. The Yangtze and Yellow Rivers, the third- and sixth-longest in the world, respectively, flow from the Tibetan Plateau to the densely populated eastern seaboard. China is a unitary one-party socialist republic and is recognized as a major global power, boasting the world's second-largest economy by nominal GDP. The Great Wall of China is one of its most famous historical landmarks.",
    "Russia": "Russia, officially the Russian Federation, is a transcontinental country spanning Eastern Europe and Northern Asia. It is the largest country in the world by land area, covering over 17 million square kilometers, and encompasses eleven time zones. Moscow is the capital and largest city, followed by Saint Petersburg, which serves as Russia's cultural center. The country's landscape includes vast plains, dense forests known as the taiga, and prominent mountain ranges such as the Urals and the Caucasus. Russia has a rich history characterized by the Russian Empire and its later role as the leading constituent of the Soviet Union. Today, it is a federal semi-presidential republic. The Russian economy is heavily reliant on its extensive natural resources, particularly oil and natural gas.",
    "Pakistan": "Pakistan, officially the Islamic Republic of Pakistan, is a country in South Asia. It is the world's fifth-most populous country, with a population of over 240 million people, and has the second-largest Muslim population. Islamabad is the nation's capital, while Karachi is its largest city and financial hub. Pakistan has a 1,046-kilometre coastline along the Arabian Sea and the Gulf of Oman in the south. The country's geography is highly diverse, featuring the towering peaks of the Karakoram and Himalayan mountain ranges in the north, including K2, the world's second-highest mountain. The Indus River flows through the length of the country, supporting agriculture and densely populated valleys. Pakistan was created in 1947 as an independent homeland for Indian Muslims following the partition of British India."
}

STOP_WORDS = set([
    "a", "about", "above", "across", "after", "afterwards", "again", "against", "all", "almost", "alone", "along", "already", "also", "although", "always", "am", "among", "amongst", "amoungst", "amount", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "around", "as", "at", "back", "be", "became", "because", "become", "becomes", "becoming", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "bill", "both", "bottom", "but", "by", "call", "can", "cannot", "cant", "co", "con", "could", "couldnt", "cry", "de", "describe", "detail", "do", "done", "down", "due", "during", "each", "eg", "eight", "either", "eleven", "else", "elsewhere", "empty", "enough", "etc", "even", "ever", "every", "everyone", "everything", "everywhere", "except", "few", "fifteen", "fify", "fill", "find", "fire", "first", "five", "for", "former", "formerly", "forty", "found", "four", "from", "front", "full", "further", "get", "give", "go", "had", "has", "hasnt", "have", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "ie", "if", "in", "inc", "indeed", "interest", "into", "is", "it", "its", "itself", "keep", "last", "latter", "latterly", "least", "less", "ltd", "made", "many", "may", "me", "meanwhile", "might", "mill", "mine", "more", "moreover", "most", "mostly", "move", "much", "must", "my", "myself", "name", "namely", "neither", "never", "nevertheless", "next", "nine", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "of", "off", "often", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "own", "part", "per", "perhaps", "please", "put", "rather", "re", "same", "see", "seem", "seemed", "seeming", "seems", "serious", "several", "she", "should", "show", "side", "since", "sincere", "six", "sixty", "so", "some", "somehow", "someone", "something", "sometime", "sometimes", "somewhere", "still", "such", "system", "take", "ten", "than", "that", "the", "their", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "thick", "thin", "third", "this", "those", "though", "three", "through", "throughout", "thru", "thus", "to", "together", "too", "top", "toward", "towards", "twelve", "twenty", "two", "un", "under", "until", "up", "upon", "us", "very", "via", "was", "we", "well", "were", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "will", "with", "within", "without", "would", "yet", "you", "your", "yours", "yourself", "yourselves"
])

def tokenize(text):
    words = re.findall(r"\b\w\w+\b", text.lower())
    return [w for w in words if w not in STOP_WORDS]

documents = list(raw_corpus.values())
columns_names = list(raw_corpus.keys())

tokenized_docs = [tokenize(doc) for doc in documents]
vocabulary = set([word for doc in tokenized_docs for word in doc])
df = {word: sum(1 for doc in tokenized_docs if word in doc) for word in vocabulary}
N = len(documents)

idf = {word: math.log((1 + N) / (1 + df[word])) + 1.0 for word in vocabulary}

def compute_tfidf_vector(tokens, vocab_filter=True):
    tf = {}
    for word in tokens:
        if vocab_filter and word not in vocabulary:
            continue
        tf[word] = tf.get(word, 0) + 1

    vector = {word: tf[word] * idf[word] for word in tf}

    norm = math.sqrt(sum(val ** 2 for val in vector.values()))
    if norm > 0:
        vector = {word: val / norm for word, val in vector.items()}
    return vector

tfidf_matrix = [compute_tfidf_vector(doc, vocab_filter=False) for doc in tokenized_docs]

new_text = "This colossal nation spans two different continents, making it a bridge between distinct cultural spheres. Because of its sheer size, traveling from its western borders to its eastern shores means passing through multiple time zones. The natural environment is notoriously harsh, featuring vast, freezing plains and endless stretches of dense, needle-leaf forests. Historically, it has transitioned through powerful imperial eras and a major twentieth-century ideological union, leaving a complex legacy. Today, its global influence is heavily sustained by the extraction and exportation of immense underground energy reserves, particularly fossil fuels, which lie hidden beneath its freezing terrain."

new_tokens = tokenize(new_text)
new_vector = compute_tfidf_vector(new_tokens, vocab_filter=True)

def cosine_sim(vec1, vec2):
    return sum(vec1.get(word, 0) * vec2.get(word, 0) for word in set(vec1) & set(vec2))

results = {name: cosine_sim(new_vector, tfidf_matrix[i]) for i, name in enumerate(columns_names)}

print("similarity score:")
for country, score in results.items():
    print(f" {country}: {score:.4f}")

best_match = max(results, key=results.get)
print(f"\n The country closest to this new text is: {best_match}\n")

russia_index = columns_names.index("Russia")
russia_vector = tfidf_matrix[russia_index]

shared_words = set(new_vector.keys()) & set(russia_vector.keys())

evidence_data = []
for word in shared_words:
    evidence_data.append({
        "Matched resonance words": word,
        "Weight in new text": round(new_vector[word], 4),
        "Weight in the original Russian text": round(russia_vector[word], 4)
    })

evidence_data.sort(key=lambda x: x["Weight in the original Russian text"], reverse=True)

print(f"{'Matched resonance words':<25} {'Weight in new text':<20} {'Weight in the original Russian text'}")
for row in evidence_data:
    print(f"{row['Matched resonance words']:<25} {row['Weight in new text']:<20.4f} {row['Weight in the original Russian text']:.4f}")

similarity score:
 China: 0.1158
 Russia: 0.3528
 Pakistan: 0.0555

 The country closest to this new text is: Russia

Matched resonance words   Weight in new text   Weight in the original Russian text
natural                   0.2521               0.2197
zones                     0.2521               0.1099
dense                     0.2521               0.1099
union                     0.2521               0.1099
particularly              0.2521               0.1099
cultural                  0.2521               0.1099
heavily                   0.2521               0.1099
today                     0.2521               0.1099
time                      0.2521               0.1099
plains                    0.2521               0.1099
forests                   0.1917               0.0836
vast                      0.1917               0.0836
eastern                   0.1917               0.0836
